# Grok-multimodal · FS13-FS14 Unified & Any-to-Any

Tiny MLLM token stream then multi-route any-to-any router.


In [ ]:
import os, json, math, random, time
from pathlib import Path
import numpy as np
os.environ.pop("CUDA_VISIBLE_DEVICES", None)
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
OUT=Path("/kaggle/working"); FIG=OUT/"figures"; RES=OUT/"results"
FIG.mkdir(parents=True, exist_ok=True); RES.mkdir(parents=True, exist_ok=True)
device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device",device,"gpus",torch.cuda.device_count() if torch.cuda.is_available() else 0)
PROGRESS={}

def make_shape_image(kind, size=32):
    img=np.ones((size,size,3),np.float32)*0.95
    yy,xx=np.mgrid[0:size,0:size]; cy,cx=size//2,size//2
    if kind=="red_circle":
        m=(yy-cy)**2+(xx-cx)**2<=(size*0.28)**2; img[m]=(0.9,0.15,0.12)
    elif kind=="blue_square":
        m=(np.abs(yy-cy)<size*0.25)&(np.abs(xx-cx)<size*0.25); img[m]=(0.15,0.25,0.85)
    elif kind=="green_triangle":
        m=(yy>cy-size*0.25)&(yy<cy+size*0.3)
        m&=np.abs(xx-cx)<(yy-(cy-size*0.25))*0.7; img[m]=(0.15,0.75,0.25)
    else:
        raise ValueError(kind)
    return img

CLASSES=["red_circle","blue_square","green_triangle"]
c2i={c:i for i,c in enumerate(CLASSES)}


## FS13 · Multimodal LM interface


In [ ]:
# Multimodal LLM-style: shared token stream with image prefix embeddings + tiny transformer LM
# Task: given image tokens + question tokens, predict answer tokens (VQA as LM)

ANS_VOCAB=["<pad>","<bos>","<eos>","red","blue","green","circle","square","triangle","yes","no"]
av={t:i for i,t in enumerate(ANS_VOCAB)}
Q_VOCAB=["<pad>","what","color","shape","is","it","a","?"]
qv={t:i for i,t in enumerate(Q_VOCAB)}

QA=[
    ("red_circle",["what","color","?"],["red"]),
    ("red_circle",["what","shape","?"],["circle"]),
    ("blue_square",["what","color","?"],["blue"]),
    ("blue_square",["is","it","a","square","?"],["yes"]),
    ("green_triangle",["what","shape","?"],["triangle"]),
    ("green_triangle",["is","it","a","circle","?"],["no"]),
]

class ImgTok(nn.Module):
    def __init__(self, n_tok=4, d=64):
        super().__init__()
        self.cnn=nn.Sequential(
            nn.Conv2d(3,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(32,d,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d((2,2)))
        self.n_tok=n_tok; self.d=d
    def forward(self,x):
        f=self.cnn(x).flatten(2).transpose(1,2)  # [B,4,d]
        return f

class TinyMLLM(nn.Module):
    def __init__(self, d=64, n_layers=2, n_heads=4):
        super().__init__()
        self.img=ImgTok(d=d)
        self.qemb=nn.Embedding(len(Q_VOCAB), d)
        self.aemb=nn.Embedding(len(ANS_VOCAB), d)
        layer=nn.TransformerEncoderLayer(d_model=d, nhead=n_heads, dim_feedforward=128, batch_first=True)
        self.tr=nn.TransformerEncoder(layer, num_layers=n_layers)
        self.head=nn.Linear(d, len(ANS_VOCAB))
        self.d=d
    def forward(self, images, qids, aids_in):
        # images [B,3,H,W], qids [B,Lq], aids_in [B,La] teacher forcing inputs
        it=self.img(images)
        qt=self.qemb(qids)
        at=self.aemb(aids_in)
        seq=torch.cat([it,qt,at],1)
        # causal mask so model cannot peek future answer tokens
        L=seq.size(1)
        causal=torch.triu(torch.ones(L,L,device=seq.device), diagonal=1).bool()
        h=self.tr(seq, mask=causal)
        La=aids_in.size(1)
        return self.head(h[:,-La:,:])

def encode_q(toks,L=6):
    ids=[qv.get(t,0) for t in toks][:L]; ids+=[0]*(L-len(ids)); return ids

def encode_a(toks,L=3):
    ids=[av["<bos>"]]+[av[t] for t in toks]+[av["<eos>"]]
    ids=ids[:L]; ids+=[av["<pad>"]]*(L-len(ids)); return ids

def batch(B=32,size=32):
    ims=[]; qs=[]; a_in=[]; a_out=[]
    for _ in range(B):
        k,q,a=random.choice(QA)
        img=make_shape_image(k,size)
        img=np.clip(img+0.03*np.random.randn(*img.shape).astype(np.float32),0,1)
        ims.append(img.transpose(2,0,1))
        qs.append(encode_q(q))
        full=encode_a(a,L=4)
        a_in.append(full[:-1]); a_out.append(full[1:])
    return (torch.tensor(np.stack(ims),dtype=torch.float32),
            torch.tensor(qs,dtype=torch.long),
            torch.tensor(a_in,dtype=torch.long),
            torch.tensor(a_out,dtype=torch.long))

mllm=TinyMLLM().to(device)
opt=torch.optim.Adam(mllm.parameters(), lr=2e-3)
hist13=[]
for epoch in range(1,50):
    mllm.train(); losses=[]; accs=[]
    for _ in range(60):
        im,q,ain,aout=batch(); im,q,ain,aout=[t.to(device) for t in (im,q,ain,aout)]
        opt.zero_grad(set_to_none=True)
        lg=mllm(im,q,ain)
        loss=F.cross_entropy(lg.reshape(-1,len(ANS_VOCAB)), aout.reshape(-1), ignore_index=av["<pad>"])
        loss.backward(); opt.step(); losses.append(loss.item())
        mask=aout!=av["<pad>"]
        accs.append((lg.argmax(-1)[mask]==aout[mask]).float().mean().item())
    row={"epoch":epoch,"loss":round(float(np.mean(losses)),4),"token_acc":round(float(np.mean(accs)),4)}
    hist13.append(row)
    if epoch%5==0: print(row)

@torch.no_grad()
def mllm_answer(kind, qtoks):
    mllm.eval()
    img=make_shape_image(kind,32)
    im=torch.tensor(img.transpose(2,0,1)[None],dtype=torch.float32,device=device)
    q=torch.tensor([encode_q(qtoks)],device=device)
    ids=[av["<bos>"]]
    outs=[]
    for _ in range(3):
        # pad to fixed La=3 teacher-force width used in training
        cur=ids + [av["<pad>"]]*(3-len(ids))
        ain=torch.tensor([cur[:3]],device=device)
        lg=mllm(im,q,ain)
        step=len(ids)-1
        nxt=int(lg[0, step].argmax().item())
        if nxt in (av["<eos>"], av["<pad>"]): break
        outs.append(ANS_VOCAB[nxt]); ids.append(nxt)
    return " ".join(outs)

rows13=[]
for k,q,a in QA:
    pred=mllm_answer(k,q)
    rows13.append({"image":k,"q":" ".join(q),"gt":" ".join(a),"pred":pred,"ok":pred.strip()==" ".join(a)})
acc13=sum(r["ok"] for r in rows13)/len(rows13)
print(rows13, acc13)
fig,ax=plt.subplots(figsize=(6,2.5))
ax.axis("off")
ax.set_title("FS13 Tiny MLLM unified token stream (image prefix + text)",fontsize=10)
tbl=ax.table(cellText=[[r["image"],r["q"],r["gt"],r["pred"],str(r["ok"])] for r in rows13],
             colLabels=["image","question","gt","pred","ok"], loc="center", cellLoc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(7); tbl.scale(1,1.2)
fig.tight_layout(); fig.savefig(FIG/"fs13_mllm.png",dpi=140); plt.close()
fs13={"stage":"FS13","method":"tiny multimodal LM: image tokens + text tokens transformer",
      "history":hist13,"acc":acc13,"rows":rows13,
      "vs_prev":"FS06 separate fusion head; FS13 single autoregressive interface like LLaVA-style",
      "figure":"figures/fs13_mllm.png"}
(RES/"fs13.json").write_text(json.dumps(fs13,indent=2)); PROGRESS["FS13"]="ok"; print("FS13 DONE")


## FS14 · Any-to-Any routing


In [ ]:
# Any-to-Any router: dispatch by input modality tags to expert heads (image/text/audio/video)
# Demonstrates unified API: route(in_modality, out_modality, payload) -> result
# Experts re-use minimal modules trained above patterns

class AnyToAny(nn.Module):
    """Router + tiny shared trunk + modality-specific adapters."""
    def __init__(self, d=64):
        super().__init__()
        self.img_in=nn.Sequential(nn.Conv2d(3,16,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Linear(16,d))
        self.txt_in=nn.Embedding(32,d)
        self.aud_in=nn.Sequential(nn.Conv1d(1,16,5,stride=2),nn.ReLU(),nn.AdaptiveAvgPool1d(1),nn.Flatten(),nn.Linear(16,d))
        self.vid_in=nn.Sequential(nn.Conv3d(3,16,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool3d(1),nn.Flatten(),nn.Linear(16,d))
        self.trunk=nn.Sequential(nn.Linear(d,d),nn.ReLU(),nn.Linear(d,d))
        # outputs
        self.out_txt=nn.Linear(d, 8)  # 8 word classes
        self.out_img=nn.Linear(d, 3*16*16)  # tiny image
        self.out_cls=nn.Linear(d, 4)
    def encode(self, modality, payload):
        if modality=="image":
            return self.img_in(payload)
        if modality=="text":
            return self.txt_in(payload).mean(1)
        if modality=="audio":
            return self.aud_in(payload)
        if modality=="video":
            return self.vid_in(payload)
        raise ValueError(modality)
    def forward(self, in_mod, out_mod, payload):
        z=self.trunk(self.encode(in_mod, payload))
        if out_mod=="text": return self.out_txt(z)
        if out_mod=="image": return self.out_img(z).view(-1,3,16,16)
        if out_mod=="label": return self.out_cls(z)
        raise ValueError(out_mod)

WORD8=["red","blue","green","circle","square","triangle","yes","no"]
w8={w:i for i,w in enumerate(WORD8)}

router=AnyToAny().to(device)
opt=torch.optim.Adam(router.parameters(), lr=3e-3)

def rand_image_batch(B=32):
    xs=[]; ys=[]
    for _ in range(B):
        k=random.choice(CLASSES)
        img=make_shape_image(k,32)
        xs.append(img.transpose(2,0,1)); ys.append(w8[k.split("_")[0]])  # color word
    return torch.tensor(np.stack(xs),dtype=torch.float32), torch.tensor(ys)

def rand_audio_batch(B=32, n=800):
    # tone by color
    freq={"red":220,"blue":440,"green":660}
    xs=[]; ys=[]
    for _ in range(B):
        color=random.choice(list(freq))
        t=np.linspace(0,0.2,n,endpoint=False)
        wav=(0.5*np.sin(2*np.pi*freq[color]*t)).astype(np.float32)
        xs.append(wav[None]); ys.append(w8[color])
    return torch.tensor(np.stack(xs),dtype=torch.float32), torch.tensor(ys)

hist14=[]
for epoch in range(1,60):
    router.train(); losses=[]
    # task A: image -> text color
    xb,yb=rand_image_batch(); xb,yb=xb.to(device),yb.to(device)
    loss1=F.cross_entropy(router("image","text",xb), yb)
    # task B: audio -> text color
    ab,ay=rand_audio_batch(); ab,ay=ab.to(device),ay.to(device)
    loss2=F.cross_entropy(router("audio","text",ab), ay)
    # task C: text -> image (reconstruct class prototype via MSE on tiny canvas)
    # encode text ids of class names
    texts=[]; imgs=[]
    for k in CLASSES:
        ids=[w8[k.split("_")[0]], w8[k.split("_")[1]]]
        texts.append(ids)
        im=make_shape_image(k,16).transpose(2,0,1)
        imgs.append(im)
    # expand
    tb=torch.tensor(texts*10,dtype=torch.long,device=device)
    ib=torch.tensor(np.stack(imgs*10),dtype=torch.float32,device=device)
    pred_img=router("text","image",tb)
    loss3=F.mse_loss(pred_img, ib)
    loss=loss1+loss2+0.5*loss3
    # extra focused steps on classification routes
    for _k in range(3):
        xb,yb=rand_image_batch(); xb,yb=xb.to(device),yb.to(device)
        loss=loss+F.cross_entropy(router("image","text",xb), yb)
        ab,ay=rand_audio_batch(); ab,ay=ab.to(device),ay.to(device)
        loss=loss+F.cross_entropy(router("audio","text",ab), ay)
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    hist14.append({"epoch":epoch,"loss":round(loss.item(),4),"l_img2txt":round(loss1.item(),4),
                   "l_aud2txt":round(loss2.item(),4),"l_txt2img":round(loss3.item(),4)})
    if epoch%5==0: print(hist14[-1])

# evaluate routes
router.eval()
routes=[]
with torch.no_grad():
    # image->text
    for k in CLASSES:
        x=torch.tensor(make_shape_image(k,32).transpose(2,0,1)[None],dtype=torch.float32,device=device)
        pred=WORD8[router("image","text",x).argmax(1).item()]
        routes.append({"route":"image->text","in":k,"out":pred,"ok":pred==k.split("_")[0]})
    # audio->text
    for color,f in [("red",220),("blue",440),("green",660)]:
        t=np.linspace(0,0.2,800,endpoint=False)
        wav=(0.5*np.sin(2*np.pi*f*t)).astype(np.float32)
        a=torch.tensor(wav[None,None],dtype=torch.float32,device=device)
        pred=WORD8[router("audio","text",a).argmax(1).item()]
        routes.append({"route":"audio->text","in":color,"out":pred,"ok":pred==color})
    # text->image
    fig,axes=plt.subplots(1,3,figsize=(7,2.5))
    for j,k in enumerate(CLASSES):
        ids=torch.tensor([[w8[k.split("_")[0]], w8[k.split("_")[1]]]],device=device)
        im=router("text","image",ids)[0].cpu().numpy().transpose(1,2,0)
        im=np.clip(im,0,1)
        axes[j].imshow(im); axes[j].set_title(f"text->{k}",fontsize=8); axes[j].axis("off")
        routes.append({"route":"text->image","in":k,"out":"image16","mean_rgb":[round(float(x),3) for x in im.mean(axis=(0,1))]})
fig.suptitle("FS14 any-to-any text->image samples"); fig.tight_layout()
fig.savefig(FIG/"fs14_any2any.png",dpi=120); plt.close()

ok_cls=[r for r in routes if "ok" in r]
acc14=sum(r["ok"] for r in ok_cls)/max(1,len(ok_cls))
fs14={"stage":"FS14","method":"any-to-any router with shared trunk + modality adapters",
      "history":hist14,"routes":routes,"route_acc":acc14,
      "vs_prev":"FS13 single VQA LM interface; FS14 multi-route in_mod x out_mod API (frontier pattern)",
      "figure":"figures/fs14_any2any.png",
      "task_coverage":{
          "audio_text":"audio->text route",
          "image_text":"image->text + FS03-06",
          "image_text_image":"text->image + FS11",
          "image_text_video":"FS12",
          "vqa":"FS06/FS13",
          "docqa":"FS07",
          "video_text":"FS09",
          "visual_doc_retrieval":"FS08",
          "any_to_any":"FS14 router",
      }}
(RES/"fs14.json").write_text(json.dumps(fs14,indent=2)); PROGRESS["FS14"]="ok"; print("FS14 DONE",acc14)

# final curriculum map write
final={
  "curriculum":"multimodal-from-scratch",
  "stages_completed":list(PROGRESS.keys()),
  "map":[
    "FS00 tensors/alignment","FS01 bag colors","FS02 CNN word","FS03 ShowTell","FS04 Attend",
    "FS05 CLIP","FS06 VQA","FS07 DocQA","FS08 VisDocRet","FS09 VideoText","FS10 ASR",
    "FS11 DiffGen","FS12 VideoGen","FS13 MLLM","FS14 Any2Any"
  ],
}
(RES/"FINAL_CURRICULUM.json").write_text(json.dumps(final,indent=2))
(OUT/"SUCCESS").write_text("ok\n")
print("FS13-14 COMPLETE", final)
